# Preprocessing — IHSG Multivariate LSTM (v2)
## Analisis Prediksi IHSG Menggunakan Multivariate LSTM Berdasarkan Data Historis dan Variabel Makroekonomi Periode 2015–2024

**Pipeline:**
1. Load & inspeksi data (IHSG, Kurs, Inflasi, BI Rate)
2. Timeline alignment (daily × monthly overlap)
3. Train-test split 80:20 berbasis cutoff tanggal
4. Scaling (MinMaxScaler, fit on train only)
5. Dual-branch sequence generation
6. Univariate sequence generation
7. Simpan artefak ke `data/processed_v2/`

**Perubahan dari v1:**
- `y_test` di-generate **sekali secara independen** dari `test_daily_scaled`, terpisah dari loop `seq_monthly`
- Semua kombinasi `(seq_daily, seq_monthly)` berbagi `y_test` yang identik untuk `seq_daily` yang sama
- Output disimpan ke `data/processed_v2/` agar tidak overwrite artefak v1


## 1. Imports & Konfigurasi

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import joblib
import json
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')
np.random.seed(42)

ROOT           = Path(os.getcwd()).parent
DATA_RAW       = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed_v2"   # ← folder baru
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print("ROOT          :", ROOT)
print("DATA_RAW      :", DATA_RAW)
print("DATA_PROCESSED:", DATA_PROCESSED)


ROOT          : d:\project_skripsi
DATA_RAW      : d:\project_skripsi\data\raw
DATA_PROCESSED: d:\project_skripsi\data\processed_v2


## 2. Konfigurasi Hyperparameter Grid Search

Pada tahap ini ditetapkan kandidat hyperparameter yang akan diuji saat pemodelan. Panjang sekuens harian (`seq_daily`) dipilih mengacu pada horizon analisis teknikal saham yang umum (±1, 3, dan 4,5 bulan trading), sedangkan panjang sekuens bulanan (`seq_monthly`) mengikuti siklus kebijakan moneter (kuartalan, setengah tahun, dan tahunan). Nilai-nilai ini disimpan di `GRID_CONFIG` agar konsisten dipakai di seluruh notebook.

In [2]:
# Kandidat hyperparameter untuk grid search di modeling.ipynb
# seq_daily  : berdasarkan horizon analisis teknikal saham yang umum digunakan
#   22  hari  ≈ 1  bulan trading
#   60  hari  ≈ 3  bulan trading
#   90  hari  ≈ 4.5 bulan trading
#
# seq_monthly: berdasarkan siklus kebijakan moneter
#   3  bulan  = 1 kuartal (periode evaluasi ekonomi standar)
#   6  bulan  = setengah tahun (horizon kebijakan moneter jangka menengah)
#   12 bulan  = 1 tahun penuh (siklus tahunan inflasi dan suku bunga)

GRID_CONFIG = {
    "seq_daily_candidates"   : [22, 60, 90],
    "seq_monthly_candidates" : [3, 6, 12],
    "batch_size_candidates"  : [16, 32],
    "grid_horizon"           : 5,
    "test_ratio"             : 0.20,
    "random_seed"            : 42,
}

print("Grid Search Config:")
for k, v in GRID_CONFIG.items():
    print(f"  {k:30s}: {v}")


Grid Search Config:
  seq_daily_candidates          : [22, 60, 90]
  seq_monthly_candidates        : [3, 6, 12]
  batch_size_candidates         : [16, 32]
  grid_horizon                  : 5
  test_ratio                    : 0.2
  random_seed                   : 42


## 3. Load Raw Data

| File | Frekuensi | Kolom | Format tanggal |
|---|---|---|---|
| `data/processed/data_ihsg.csv` | Harian | date, ihsg | YYYY-MM-DD |
| `data/raw/nilai_tukar.csv` | Harian | date, nilai_tukar | DD-MM-YYYY |
| `data/raw/inflasi_monthly.csv` | Bulanan | date, inflasi | YYYY-MM-DD |
| `data/raw/bi_rate_monthly.csv` | Bulanan | date, bi_rate | YYYY-MM-DD |


In [3]:
# IHSG dari data/processed/ (sudah bersih dari preprocessing univariate)
DATA_PROCESSED_V1 = ROOT / "data" / "processed"

df_ihsg = pd.read_csv(DATA_PROCESSED_V1 / "data_ihsg.csv")
df_ihsg['date'] = pd.to_datetime(df_ihsg['date'])
df_ihsg = df_ihsg.sort_values('date').reset_index(drop=True)
print("IHSG shape:", df_ihsg.shape)
print(df_ihsg.head(3))
print()

df_kurs = pd.read_csv(DATA_RAW / "nilai_tukar.csv")
df_kurs['date'] = pd.to_datetime(df_kurs['date'], format='%d-%m-%Y')
df_kurs = df_kurs.sort_values('date').reset_index(drop=True)
print("Kurs shape:", df_kurs.shape)
print(df_kurs.head(3))
print()

df_inflasi = pd.read_csv(DATA_RAW / "inflasi_monthly.csv")
df_inflasi['date'] = pd.to_datetime(df_inflasi['date'])
df_inflasi = df_inflasi.sort_values('date').reset_index(drop=True)
print("Inflasi shape:", df_inflasi.shape)
print(df_inflasi.head(3))
print()

df_birate = pd.read_csv(DATA_RAW / "bi_rate_monthly.csv")
df_birate['date'] = pd.to_datetime(df_birate['date'])
df_birate = df_birate.sort_values('date').reset_index(drop=True)
print("BI Rate shape:", df_birate.shape)
print(df_birate.head(3))


IHSG shape: (2426, 2)
        date     ihsg
0 2015-01-02  5242.77
1 2015-01-05  5219.99
2 2015-01-06  5169.06

Kurs shape: (2554, 2)
        date  nilai_tukar
0 2015-01-01      12385.0
1 2015-01-02      12542.5
2 2015-01-05      12627.5

Inflasi shape: (120, 2)
        date  inflasi
0 2015-01-01    -0.24
1 2015-02-01    -0.36
2 2015-03-01     0.17

BI Rate shape: (120, 2)
        date  bi_rate
0 2015-01-01     7.75
1 2015-02-01     7.50
2 2015-03-01     7.50


## 4. Inspeksi & Validasi Data

Sebelum data diolah lebih jauh, dilakukan pemeriksaan kualitas untuk memastikan setiap sumber bersih dan layak digunakan: cek nilai hilang (*missing value*), rentang tanggal tiap variabel, duplikat tanggal, serta statistik dasar. Langkah ini penting agar masalah data tidak terbawa ke tahap pemodelan dan menimbulkan bias.

In [4]:
print("=" * 50)
print("Missing Values Check")
print("=" * 50)
for name, df in [("IHSG", df_ihsg), ("Kurs", df_kurs),
                  ("Inflasi", df_inflasi), ("BI Rate", df_birate)]:
    mv = df.isnull().sum().sum()
    print(f"  {name:10s}: {mv} missing values")

print()
print("=" * 50)
print("Rentang Tanggal")
print("=" * 50)
for name, df in [("IHSG", df_ihsg), ("Kurs", df_kurs),
                  ("Inflasi", df_inflasi), ("BI Rate", df_birate)]:
    print(f"  {name:10s}: {df['date'].min().date()} → {df['date'].max().date()}  ({len(df)} baris)")

print()
print("=" * 50)
print("Duplikat Tanggal")
print("=" * 50)
for name, df in [("IHSG", df_ihsg), ("Kurs", df_kurs),
                  ("Inflasi", df_inflasi), ("BI Rate", df_birate)]:
    dup = df['date'].duplicated().sum()
    print(f"  {name:10s}: {dup} duplikat")

print()
print("=" * 50)
print("Statistik Dasar")
print("=" * 50)
for name, df in [("IHSG", df_ihsg), ("Kurs", df_kurs),
                  ("Inflasi", df_inflasi), ("BI Rate", df_birate)]:
    num_cols = df.select_dtypes(include='number').columns
    print(f"\n{name}:")
    print(df[num_cols].describe().round(4))


Missing Values Check
  IHSG      : 0 missing values
  Kurs      : 0 missing values
  Inflasi   : 0 missing values
  BI Rate   : 0 missing values

Rentang Tanggal
  IHSG      : 2015-01-02 → 2024-12-30  (2426 baris)
  Kurs      : 2015-01-01 → 2024-12-31  (2554 baris)
  Inflasi   : 2015-01-01 → 2024-12-01  (120 baris)
  BI Rate   : 2015-01-01 → 2024-12-01  (120 baris)

Duplikat Tanggal
  IHSG      : 0 duplikat
  Kurs      : 0 duplikat
  Inflasi   : 0 duplikat
  BI Rate   : 0 duplikat

Statistik Dasar

IHSG:
            ihsg
count  2426.0000
mean   6063.9316
std     844.2708
min    3937.6300
25%    5386.1125
50%    6107.3800
75%    6786.6200
max    7905.3900

Kurs:
       nilai_tukar
count    2554.0000
mean    14313.5621
std       890.3751
min     12385.0000
25%     13543.1250
50%     14243.5000
75%     14905.0000
max     16575.0000

Inflasi:
        inflasi
count  120.0000
mean     0.2396
std      0.2955
min     -0.4500
25%      0.0375
50%      0.2000
75%      0.4025
max      1.1700

BI R

## 5. Timeline Alignment

Data berasal dari dua frekuensi berbeda — harian (IHSG, kurs) dan bulanan (inflasi, BI Rate) — sehingga perlu diselaraskan terlebih dahulu agar setiap observasi benar-benar sepadan. Penyelarasan dilakukan dengan *inner join* sehingga hanya menyisakan periode di mana seluruh variabel pada masing-masing cabang tersedia.

- **Cabang harian (daily):** *inner join* IHSG × kurs, hanya mengambil hari bursa di mana keduanya tersedia.
- **Cabang bulanan (monthly):** *inner join* inflasi × BI Rate per bulan.

In [5]:
# ── Daily: IHSG × Kurs ──
df_daily = pd.merge(df_ihsg, df_kurs, on='date', how='inner')
df_daily = df_daily.sort_values('date').reset_index(drop=True)
df_daily = df_daily[['date', 'ihsg', 'nilai_tukar']]
assert df_daily.isnull().sum().sum() == 0, "Masih ada NaN di df_daily!"
print(f"Daily dataset shape: {df_daily.shape}")
print(f"Rentang: {df_daily['date'].min().date()} → {df_daily['date'].max().date()}")
print(df_daily.head())


Daily dataset shape: (2426, 3)
Rentang: 2015-01-02 → 2024-12-30
        date     ihsg  nilai_tukar
0 2015-01-02  5242.77      12542.5
1 2015-01-05  5219.99      12627.5
2 2015-01-06  5169.06      12657.5
3 2015-01-07  5207.12      12738.5
4 2015-01-08  5211.83      12680.0


In [6]:
# ── Monthly: Inflasi × BI Rate ──
df_inflasi['month'] = df_inflasi['date'].dt.to_period('M')
df_birate['month']  = df_birate['date'].dt.to_period('M')

df_monthly = pd.merge(
    df_inflasi[['month', 'inflasi']],
    df_birate[['month', 'bi_rate']],
    on='month', how='inner'
)
df_monthly = df_monthly.sort_values('month').reset_index(drop=True)
df_monthly['date'] = df_monthly['month'].dt.to_timestamp()
df_monthly = df_monthly[['date', 'inflasi', 'bi_rate']]
assert df_monthly.isnull().sum().sum() == 0, "Masih ada NaN di df_monthly!"
print(f"Monthly dataset shape: {df_monthly.shape}")
print(f"Rentang: {df_monthly['date'].min().date()} → {df_monthly['date'].max().date()}")
print(df_monthly.head())


Monthly dataset shape: (120, 3)
Rentang: 2015-01-01 → 2024-12-01
        date  inflasi  bi_rate
0 2015-01-01    -0.24     7.75
1 2015-02-01    -0.36     7.50
2 2015-03-01     0.17     7.50
3 2015-04-01     0.36     7.50
4 2015-05-01     0.50     7.50


## 6. Penyelarasan Periode Overlap Daily & Monthly

Setelah masing-masing cabang terbentuk, kedua dataset dipotong agar hanya mencakup rentang bulan yang beririsan (*overlap*) antara data harian dan bulanan. Tujuannya memastikan setiap sekuens harian selalu memiliki pasangan konteks bulanan yang valid, sehingga tidak ada observasi yang menggantung tanpa variabel makroekonomi pendampingnya.

In [7]:
daily_start_month  = df_daily['date'].min().to_period('M')
daily_end_month    = df_daily['date'].max().to_period('M')
monthly_start      = df_monthly['date'].min().to_period('M')
monthly_end        = df_monthly['date'].max().to_period('M')

overlap_start = max(daily_start_month, monthly_start)
overlap_end   = min(daily_end_month, monthly_end)

print(f"Daily   : {daily_start_month} → {daily_end_month}")
print(f"Monthly : {monthly_start} → {monthly_end}")
print(f"Overlap : {overlap_start} → {overlap_end}")

df_daily = df_daily[
    (df_daily['date'].dt.to_period('M') >= overlap_start) &
    (df_daily['date'].dt.to_period('M') <= overlap_end)
].reset_index(drop=True)

df_monthly = df_monthly[
    (df_monthly['date'].dt.to_period('M') >= overlap_start) &
    (df_monthly['date'].dt.to_period('M') <= overlap_end)
].reset_index(drop=True)

print(f"\nSetelah align:")
print(f"  df_daily   shape: {df_daily.shape}")
print(f"  df_monthly shape: {df_monthly.shape}")


Daily   : 2015-01 → 2024-12
Monthly : 2015-01 → 2024-12
Overlap : 2015-01 → 2024-12

Setelah align:
  df_daily   shape: (2426, 3)
  df_monthly shape: (120, 3)


## 7. Train-Test Split (80:20, Time Series)

Karena ini data deret waktu, pembagian data **tidak dilakukan secara acak** melainkan berdasarkan tanggal *cutoff*: 80% data terawal menjadi data latih dan 20% sisanya menjadi data uji. Urutan kronologis dijaga agar model tidak "mengintip" masa depan — data uji selalu berada setelah data latih. *Cutoff* harian kemudian dipakai sebagai acuan untuk membagi data bulanan pada bulan yang sama.

In [8]:
TEST_RATIO    = GRID_CONFIG['test_ratio']
n_daily       = len(df_daily)
n_daily_test  = int(np.ceil(n_daily * TEST_RATIO))
n_daily_train = n_daily - n_daily_test
cutoff_date   = df_daily.iloc[n_daily_train]['date']

df_daily_train = df_daily.iloc[:n_daily_train].copy()
df_daily_test  = df_daily.iloc[n_daily_train:].copy()

print(f"Daily total : {n_daily} baris")
print(f"  Train     : {len(df_daily_train)} baris  ({df_daily_train['date'].min().date()} → {df_daily_train['date'].max().date()})")
print(f"  Test      : {len(df_daily_test)}  baris  ({df_daily_test['date'].min().date()} → {df_daily_test['date'].max().date()})")
print(f"  Cutoff    : {cutoff_date.date()}")

cutoff_month     = cutoff_date.to_period('M')
df_monthly_train = df_monthly[df_monthly['date'].dt.to_period('M') < cutoff_month].copy()
df_monthly_test  = df_monthly[df_monthly['date'].dt.to_period('M') >= cutoff_month].copy()

print(f"\nMonthly total: {len(df_monthly)} baris")
print(f"  Train      : {len(df_monthly_train)} baris  ({df_monthly_train['date'].min().date()} → {df_monthly_train['date'].max().date()})")
print(f"  Test       : {len(df_monthly_test)}  baris  ({df_monthly_test['date'].min().date()} → {df_monthly_test['date'].max().date()})")


Daily total : 2426 baris
  Train     : 1940 baris  (2015-01-02 → 2022-12-20)
  Test      : 486  baris  (2022-12-21 → 2024-12-30)
  Cutoff    : 2022-12-21

Monthly total: 120 baris
  Train      : 95 baris  (2015-01-01 → 2022-11-01)
  Test       : 25  baris  (2022-12-01 → 2024-12-01)


## 8. Scaling (MinMaxScaler, fit on train only)

Seluruh fitur dinormalisasi ke rentang [0, 1] dengan `MinMaxScaler` agar skala antarvariabel seragam dan proses pelatihan LSTM lebih stabil. Untuk mencegah **kebocoran data (*data leakage*)**, scaler hanya di-`fit` pada data latih, lalu parameter yang sama diterapkan (`transform`) ke data uji. Dengan begitu informasi dari data uji tidak pernah ikut membentuk parameter penskalaan. Nilai hasil skala pada data uji wajar bila sedikit melampaui 1,0, yaitu ketika IHSG uji lebih tinggi dari nilai maksimum saat pelatihan.

In [9]:
DAILY_FEATURES   = ['ihsg', 'nilai_tukar']
MONTHLY_FEATURES = ['inflasi', 'bi_rate']
TARGET_COL       = 'ihsg'
target_idx       = DAILY_FEATURES.index(TARGET_COL)

# ── Scaler Daily ──
scaler_daily         = MinMaxScaler(feature_range=(0, 1))
train_daily_scaled   = scaler_daily.fit_transform(df_daily_train[DAILY_FEATURES].values)
test_daily_scaled    = scaler_daily.transform(df_daily_test[DAILY_FEATURES].values)

# ── Scaler Monthly ──
scaler_monthly       = MinMaxScaler(feature_range=(0, 1))
train_monthly_scaled = scaler_monthly.fit_transform(df_monthly_train[MONTHLY_FEATURES].values)
test_monthly_scaled  = scaler_monthly.transform(df_monthly_test[MONTHLY_FEATURES].values)

# ── Scaler Target (IHSG saja) ──
scaler_target        = MinMaxScaler(feature_range=(0, 1))
scaler_target.fit(df_daily_train[[TARGET_COL]].values)

print(f"train_daily_scaled   shape : {train_daily_scaled.shape}")
print(f"test_daily_scaled    shape : {test_daily_scaled.shape}")
print(f"train_monthly_scaled shape : {train_monthly_scaled.shape}")
print(f"test_monthly_scaled  shape : {test_monthly_scaled.shape}")
print()
print(f"IHSG train range (asli)    : [{df_daily_train['ihsg'].min():.0f}, {df_daily_train['ihsg'].max():.0f}]")
print(f"IHSG test  range (asli)    : [{df_daily_test['ihsg'].min():.0f}, {df_daily_test['ihsg'].max():.0f}]")
print()
print(f"test_daily_scaled IHSG range: [{test_daily_scaled[:,0].min():.4f}, {test_daily_scaled[:,0].max():.4f}]")
print("  Note: nilai >1.0 normal jika IHSG test melampaui max training")


train_daily_scaled   shape : (1940, 2)
test_daily_scaled    shape : (486, 2)
train_monthly_scaled shape : (95, 2)
test_monthly_scaled  shape : (25, 2)

IHSG train range (asli)    : [3938, 7318]
IHSG test  range (asli)    : [6566, 7905]

test_daily_scaled IHSG range: [0.7775, 1.1738]
  Note: nilai >1.0 normal jika IHSG test melampaui max training


## 9. Dual-Branch Sequence Generation

Data diubah menjadi pasangan sekuens untuk arsitektur *dual-branch*: setiap sampel terdiri atas jendela sekuens harian (`seq_daily` hari terakhir) dan jendela sekuens bulanan (`seq_monthly` bulan terakhir) yang menjadi konteks, dengan target IHSG satu langkah ke depan (hari t+1). Kedua cabang inilah yang nantinya digabung di dalam model.

**Perbaikan v2:** `y_test` di-generate sekali secara independen dari `test_daily_scaled` dan tidak dipengaruhi oleh `seq_monthly`. Dengan begitu `y_test` identik untuk semua kombinasi `(seq_daily, seq_monthly)` yang memiliki `seq_daily` sama, sehingga perbandingan metrik antar nilai `seq_monthly` menjadi adil.

In [10]:
def create_sequences_dual(
    daily_data_scaled,
    monthly_data_scaled,
    daily_dates,
    monthly_dates,
    seq_daily,
    seq_monthly,
    target_idx=0,
):
    """
    Generate dual-branch sequences untuk LSTM.

    Returns
    -------
    X_daily    : np.array (N, seq_daily, n_daily_features)
    X_monthly  : np.array (N, seq_monthly, n_monthly_features)
    y          : np.array (N,) — nilai target hari t+1 (scaled)
    valid_dates: list[datetime] — tanggal t+1 (tanggal target)
    """
    X_daily_list   = []
    X_monthly_list = []
    y_list         = []
    valid_dates    = []

    monthly_period_to_idx = {}
    for i, d in enumerate(monthly_dates):
        key = (pd.Timestamp(d).year, pd.Timestamp(d).month)
        monthly_period_to_idx[key] = i

    T_daily = len(daily_data_scaled)

    for t in range(seq_daily, T_daily - 1):
        x_d          = daily_data_scaled[t - seq_daily : t]
        current_date = pd.Timestamp(daily_dates[t])
        cur_year, cur_month = current_date.year, current_date.month
        cur_month_key = (cur_year, cur_month)

        if cur_month_key not in monthly_period_to_idx:
            continue
        cur_month_idx = monthly_period_to_idx[cur_month_key]
        if cur_month_idx < seq_monthly:
            continue

        x_m = monthly_data_scaled[cur_month_idx - seq_monthly : cur_month_idx]
        if x_m.shape[0] != seq_monthly:
            continue

        y_val = daily_data_scaled[t + 1, target_idx]

        X_daily_list.append(x_d)
        X_monthly_list.append(x_m)
        y_list.append(y_val)
        valid_dates.append(daily_dates[t + 1])

    if len(X_daily_list) == 0:
        raise ValueError("Tidak ada sequence yang valid! Cek seq_daily dan seq_monthly.")

    return (
        np.array(X_daily_list, dtype=np.float32),
        np.array(X_monthly_list, dtype=np.float32),
        np.array(y_list, dtype=np.float32),
        valid_dates,
    )


def create_y_test_canonical(test_daily_scaled, train_daily_scaled, seq_daily, target_idx=0):
    """
    Generate y_test secara independen dari seq_monthly.

    Menggunakan tail train (seq_daily baris terakhir) + test data
    untuk memastikan window tidak putus, lalu extract y dari index
    seq_daily..end — identik untuk semua seq_monthly dengan seq_daily yang sama.

    Returns
    -------
    y_test     : np.array (N,) — seluruh nilai target test period (scaled)
    dates_test : list[datetime] — tanggal untuk tiap y_test
    """
    tail_daily        = np.vstack([train_daily_scaled[-seq_daily:], test_daily_scaled])
    n_test            = len(test_daily_scaled)
    # y mulai dari index seq_daily (hari pertama test) sampai akhir
    # t = seq_daily     → y = tail_daily[seq_daily+1] = test_daily_scaled[1]
    # t = seq_daily+n_test-2 → y = tail_daily[-1] = test_daily_scaled[-1]
    y_test  = tail_daily[seq_daily + 1 : seq_daily + n_test, target_idx]
    return np.array(y_test, dtype=np.float32)


print("✓ create_sequences_dual()")
print("✓ create_y_test_canonical()  ← baru di v2")


✓ create_sequences_dual()
✓ create_y_test_canonical()  ← baru di v2


## 10. Generate Sequences untuk Semua Kombinasi Grid Search

Sekuens dibangkitkan untuk setiap kombinasi `(seq_daily, seq_monthly)` pada grid, agar siap dipakai langsung saat pencarian hyperparameter di tahap pemodelan.

**Perbaikan v2:** `y_test` dihitung secara *canonical* (independen dari `seq_monthly`), lalu di-*assign* ke semua kombinasi yang memiliki `seq_daily` sama sehingga target uji konsisten.

In [11]:
daily_train_dates   = df_daily_train['date'].values
daily_test_dates    = df_daily_test['date'].values
monthly_train_dates = df_monthly_train['date'].values
monthly_test_dates  = df_monthly_test['date'].values

# ── Pre-compute y_test canonical per seq_daily ──
# Satu y_test per seq_daily, identik untuk semua seq_monthly
y_test_canonical = {}
for seq_d in GRID_CONFIG['seq_daily_candidates']:
    y_test_canonical[seq_d] = create_y_test_canonical(
        test_daily_scaled, train_daily_scaled, seq_d, target_idx=target_idx
    )
    print(f"y_test canonical seq_daily={seq_d:2d}: "
          f"shape={y_test_canonical[seq_d].shape}, "
          f"range=[{y_test_canonical[seq_d].min():.4f}, {y_test_canonical[seq_d].max():.4f}]")

print()
print("✓ y_test canonical konsisten per seq_daily — tidak dipengaruhi seq_monthly")


y_test canonical seq_daily=22: shape=(485,), range=[0.7775, 1.1738]
y_test canonical seq_daily=60: shape=(485,), range=[0.7775, 1.1738]
y_test canonical seq_daily=90: shape=(485,), range=[0.7775, 1.1738]

✓ y_test canonical konsisten per seq_daily — tidak dipengaruhi seq_monthly


In [12]:
sequences_dict = {}

total_combos = (len(GRID_CONFIG['seq_daily_candidates']) *
                len(GRID_CONFIG['seq_monthly_candidates']))
combo_count  = 0

for seq_d in GRID_CONFIG['seq_daily_candidates']:
    for seq_m in GRID_CONFIG['seq_monthly_candidates']:
        combo_count += 1
        key = (seq_d, seq_m)

        try:
            # ── TRAIN ──
            X_d_tr, X_m_tr, y_tr, dates_tr = create_sequences_dual(
                train_daily_scaled, train_monthly_scaled,
                daily_train_dates, monthly_train_dates,
                seq_daily=seq_d, seq_monthly=seq_m
            )

            # ── TEST: X_daily & X_monthly tetap per-kombinasi ──
            tail_daily_scaled   = np.vstack([train_daily_scaled[-seq_d:], test_daily_scaled])
            tail_daily_dates    = np.concatenate([daily_train_dates[-seq_d:], daily_test_dates])
            tail_monthly_scaled = np.vstack([train_monthly_scaled[-seq_m:], test_monthly_scaled])
            tail_monthly_dates  = np.concatenate([monthly_train_dates[-seq_m:], monthly_test_dates])

            X_d_te, X_m_te, _, dates_te = create_sequences_dual(
                tail_daily_scaled, tail_monthly_scaled,
                tail_daily_dates, tail_monthly_dates,
                seq_daily=seq_d, seq_monthly=seq_m
            )

            # ── y_test: pakai canonical (independen dari seq_monthly) ──
            y_te = y_test_canonical[seq_d]

            # Pastikan panjang X dan y konsisten
            # X_d_te mungkin lebih panjang dari y_te karena beda titik awal skip
            # Selaraskan dengan ambil ekor X sesuai panjang y_te
            if len(X_d_te) > len(y_te):
                X_d_te   = X_d_te[-len(y_te):]
                X_m_te   = X_m_te[-len(y_te):]
                dates_te = dates_te[-len(y_te):]
            elif len(X_d_te) < len(y_te):
                y_te     = y_te[-len(X_d_te):]

            sequences_dict[key] = {
                'X_daily_train'   : X_d_tr,
                'X_monthly_train' : X_m_tr,
                'y_train'         : y_tr,
                'dates_train'     : dates_tr,
                'X_daily_test'    : X_d_te,
                'X_monthly_test'  : X_m_te,
                'y_test'          : y_te,
                'dates_test'      : dates_te,
            }

            print(f"[{combo_count}/{total_combos}] seq_daily={seq_d:3d}, seq_monthly={seq_m:2d}"
                  f"  Train: X_d={X_d_tr.shape}, X_m={X_m_tr.shape}, y={y_tr.shape}"
                  f"  | Test: X_d={X_d_te.shape}, X_m={X_m_te.shape}, y={y_te.shape}"
                  f"  y_range=[{y_te.min():.4f},{y_te.max():.4f}]")

        except ValueError as e:
            print(f"[{combo_count}/{total_combos}] seq_daily={seq_d:3d}, seq_monthly={seq_m:2d}"
                  f"  → SKIP: {e}")

print(f"\nTotal kombinasi berhasil: {len(sequences_dict)} / {total_combos}")

# Verifikasi: y_test harus identik untuk seq_daily yang sama
print("\n── Verifikasi konsistensi y_test ──")
for seq_d in GRID_CONFIG['seq_daily_candidates']:
    keys_same_seqd = [(seq_d, seq_m) for seq_m in GRID_CONFIG['seq_monthly_candidates']
                       if (seq_d, seq_m) in sequences_dict]
    if len(keys_same_seqd) > 1:
        y_ref = sequences_dict[keys_same_seqd[0]]['y_test']
        all_same = all(
            np.allclose(sequences_dict[k]['y_test'], y_ref)
            for k in keys_same_seqd[1:]
        )
        status = "✓ IDENTIK" if all_same else "✗ BERBEDA — cek bug!"
        print(f"  seq_daily={seq_d}: {status}  (shape={y_ref.shape}, "
              f"range=[{y_ref.min():.4f},{y_ref.max():.4f}])")


[1/9] seq_daily= 22, seq_monthly= 3  Train: X_d=(1864, 22, 2), X_m=(1864, 3, 2), y=(1864,)  | Test: X_d=(485, 22, 2), X_m=(485, 3, 2), y=(485,)  y_range=[0.7775,1.1738]
[2/9] seq_daily= 22, seq_monthly= 6  Train: X_d=(1803, 22, 2), X_m=(1803, 6, 2), y=(1803,)  | Test: X_d=(485, 22, 2), X_m=(485, 6, 2), y=(485,)  y_range=[0.7775,1.1738]
[3/9] seq_daily= 22, seq_monthly=12  Train: X_d=(1682, 22, 2), X_m=(1682, 12, 2), y=(1682,)  | Test: X_d=(485, 22, 2), X_m=(485, 12, 2), y=(485,)  y_range=[0.7775,1.1738]
[4/9] seq_daily= 60, seq_monthly= 3  Train: X_d=(1864, 60, 2), X_m=(1864, 3, 2), y=(1864,)  | Test: X_d=(485, 60, 2), X_m=(485, 3, 2), y=(485,)  y_range=[0.7775,1.1738]
[5/9] seq_daily= 60, seq_monthly= 6  Train: X_d=(1803, 60, 2), X_m=(1803, 6, 2), y=(1803,)  | Test: X_d=(485, 60, 2), X_m=(485, 6, 2), y=(485,)  y_range=[0.7775,1.1738]
[6/9] seq_daily= 60, seq_monthly=12  Train: X_d=(1682, 60, 2), X_m=(1682, 12, 2), y=(1682,)  | Test: X_d=(485, 60, 2), X_m=(485, 12, 2), y=(485,)  y_rang

## 11. Sequences untuk Baseline Univariate (IHSG saja)

Sebagai pembanding, dibangkitkan pula sekuens univariate yang hanya memakai IHSG tanpa variabel makroekonomi. Baseline ini menjadi acuan untuk menilai apakah penambahan variabel makro pada model multivariate benar-benar memperbaiki prediksi.

In [13]:
def create_sequences_univariate(data_scaled, seq_length, target_idx=0):
    """Generate sequences 1D untuk LSTM univariate."""
    X_list, y_list = [], []
    for t in range(seq_length, len(data_scaled) - 1):
        X_list.append(data_scaled[t - seq_length : t, target_idx : target_idx + 1])
        y_list.append(data_scaled[t + 1, target_idx])
    return (
        np.array(X_list, dtype=np.float32),
        np.array(y_list, dtype=np.float32)
    )

scaler_uni       = MinMaxScaler(feature_range=(0, 1))
train_uni_scaled = scaler_uni.fit_transform(df_daily_train[['ihsg']].values)
test_uni_scaled  = scaler_uni.transform(df_daily_test[['ihsg']].values)

sequences_uni = {}
for seq_d in GRID_CONFIG['seq_daily_candidates']:
    X_tr, y_tr = create_sequences_univariate(train_uni_scaled, seq_d)
    tail_uni   = np.vstack([train_uni_scaled[-seq_d:], test_uni_scaled])
    X_te, y_te = create_sequences_univariate(tail_uni, seq_d)

    sequences_uni[seq_d] = {
        'X_train': X_tr, 'y_train': y_tr,
        'X_test' : X_te, 'y_test' : y_te,
    }
    print(f"seq_daily={seq_d}: Train X={X_tr.shape}, y={y_tr.shape}"
          f"  | Test X={X_te.shape}, y={y_te.shape}"
          f"  y_range=[{y_te.min():.4f},{y_te.max():.4f}]")

print("\n✓ sequences_uni OK")


seq_daily=22: Train X=(1917, 22, 1), y=(1917,)  | Test X=(485, 22, 1), y=(485,)  y_range=[0.7775,1.1738]
seq_daily=60: Train X=(1879, 60, 1), y=(1879,)  | Test X=(485, 60, 1), y=(485,)  y_range=[0.7775,1.1738]
seq_daily=90: Train X=(1849, 90, 1), y=(1849,)  | Test X=(485, 90, 1), y=(485,)  y_range=[0.7775,1.1738]

✓ sequences_uni OK


## 12. Simpan Artefak Preprocessing

Seluruh hasil preprocessing — scaler, kumpulan sekuens, DataFrame train/test, dan metadata — disimpan ke `data/processed_v2/`. Artefak inilah yang akan dimuat ulang pada notebook pemodelan, sehingga tahap preprocessing tidak perlu diulang.

In [14]:
joblib.dump(scaler_daily,   DATA_PROCESSED / "scaler_daily.joblib")
joblib.dump(scaler_monthly, DATA_PROCESSED / "scaler_monthly.joblib")
joblib.dump(scaler_target,  DATA_PROCESSED / "scaler_target.joblib")
joblib.dump(scaler_uni,     DATA_PROCESSED / "scaler_univariate.joblib")
print("✓ Scalers disimpan")

joblib.dump(sequences_dict, DATA_PROCESSED / "sequences_dual.joblib")
joblib.dump(sequences_uni,  DATA_PROCESSED / "sequences_uni.joblib")
print("✓ Sequences disimpan")

df_daily_train.to_csv(DATA_PROCESSED / "df_daily_train.csv",   index=False)
df_daily_test.to_csv( DATA_PROCESSED / "df_daily_test.csv",    index=False)
df_monthly_train.to_csv(DATA_PROCESSED / "df_monthly_train.csv", index=False)
df_monthly_test.to_csv( DATA_PROCESSED / "df_monthly_test.csv",  index=False)
print("✓ DataFrames disimpan")

seq_shapes_info = {}
for (sd, sm), v in sequences_dict.items():
    seq_shapes_info[f"d{sd}_m{sm}"] = {
        "seq_daily"       : sd, "seq_monthly"      : sm,
        "X_daily_train"   : list(v['X_daily_train'].shape),
        "X_monthly_train" : list(v['X_monthly_train'].shape),
        "y_train"         : list(v['y_train'].shape),
        "X_daily_test"    : list(v['X_daily_test'].shape),
        "X_monthly_test"  : list(v['X_monthly_test'].shape),
        "y_test"          : list(v['y_test'].shape),
    }

meta = {
    "version"              : "v2",
    "note"                 : "y_test di-generate canonical (independen dari seq_monthly)",
    "n_daily_total"        : len(df_daily),
    "n_daily_train"        : len(df_daily_train),
    "n_daily_test"         : len(df_daily_test),
    "n_monthly_total"      : len(df_monthly),
    "n_monthly_train"      : len(df_monthly_train),
    "n_monthly_test"       : len(df_monthly_test),
    "cutoff_date"          : str(cutoff_date.date()),
    "daily_train_range"    : [str(df_daily_train['date'].min().date()),
                               str(df_daily_train['date'].max().date())],
    "daily_test_range"     : [str(df_daily_test['date'].min().date()),
                               str(df_daily_test['date'].max().date())],
    "daily_features"       : DAILY_FEATURES,
    "monthly_features"     : MONTHLY_FEATURES,
    "target_col"           : TARGET_COL,
    "target_idx_in_daily"  : target_idx,
    "grid_config"          : GRID_CONFIG,
    "available_dual_keys"  : [f"({sd}, {sm})" for (sd, sm) in sequences_dict.keys()],
    "sequences_shapes"     : seq_shapes_info,
    "y_test_canonical"     : {
        str(seq_d): {
            "shape": list(y_test_canonical[seq_d].shape),
            "range": [float(y_test_canonical[seq_d].min()),
                      float(y_test_canonical[seq_d].max())]
        }
        for seq_d in GRID_CONFIG['seq_daily_candidates']
    }
}

with open(DATA_PROCESSED / "preprocessing_meta.json", "w") as f:
    json.dump(meta, f, indent=2)
print("✓ preprocessing_meta.json")

print()
print("=" * 60)
print("PREPROCESSING v2 SELESAI")
print(f"  Output: {DATA_PROCESSED}")
print("=" * 60)
for p in sorted(DATA_PROCESSED.iterdir()):
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:40s} {size_kb:8.1f} KB")


✓ Scalers disimpan
✓ Sequences disimpan
✓ DataFrames disimpan
✓ preprocessing_meta.json

PREPROCESSING v2 SELESAI
  Output: d:\project_skripsi\data\processed_v2
  df_daily_test.csv                            13.3 KB
  df_daily_train.csv                           52.9 KB
  df_monthly_test.csv                           0.6 KB
  df_monthly_train.csv                          2.0 KB
  preprocessing_meta.json                       5.6 KB
  scaler_daily.joblib                           0.7 KB
  scaler_monthly.joblib                         0.7 KB
  scaler_target.joblib                          0.7 KB
  scaler_univariate.joblib                      0.7 KB
  sequences_dual.joblib                     11715.8 KB
  sequences_uni.joblib                       1609.7 KB


## 13. Verifikasi Final

Sebagai langkah penutup, artefak yang baru disimpan dimuat kembali dan diperiksa (*sanity check*): dimensi scaler, bentuk dan rentang sekuens, konsistensi `y_test` per `seq_daily`, serta isi metadata. Tujuannya memastikan semua keluaran valid dan siap dipakai pada tahap pemodelan.

In [15]:
print("=== SANITY CHECK v2 ===")

scaler_d_check = joblib.load(DATA_PROCESSED / "scaler_daily.joblib")
print(f"✓ scaler_daily   : n_features={scaler_d_check.n_features_in_}, "
      f"range=[{scaler_d_check.data_min_}, {scaler_d_check.data_max_}]")

scaler_t_check = joblib.load(DATA_PROCESSED / "scaler_target.joblib")
print(f"✓ scaler_target  : IHSG train range=[{scaler_t_check.data_min_[0]:.0f}, {scaler_t_check.data_max_[0]:.0f}]")

seq_dual_check = joblib.load(DATA_PROCESSED / "sequences_dual.joblib")
print(f"\n✓ sequences_dual : {len(seq_dual_check)} kombinasi")
print(f"  {'Key':<12} {'X_daily_te':>12} {'X_monthly_te':>14} {'y_test':>10} {'y_range':>20}")
print(f"  {'-'*70}")
for key, v in seq_dual_check.items():
    y = v['y_test']
    print(f"  {str(key):<12} {str(v['X_daily_test'].shape):>12} "
          f"{str(v['X_monthly_test'].shape):>14} {str(y.shape):>10} "
          f"[{y.min():.4f}, {y.max():.4f}]")

# Verifikasi konsistensi akhir
print("\n── Konsistensi y_test per seq_daily ──")
from itertools import groupby
for seq_d in [22, 60, 90]:
    keys = [(seq_d, sm) for sm in [3, 6, 12] if (seq_d, sm) in seq_dual_check]
    if len(keys) > 1:
        y_ref  = seq_dual_check[keys[0]]['y_test']
        ok     = all(np.allclose(seq_dual_check[k]['y_test'], y_ref) for k in keys[1:])
        status = "✓ IDENTIK" if ok else "✗ BERBEDA"
        print(f"  seq_daily={seq_d}: {status}")

seq_uni_check = joblib.load(DATA_PROCESSED / "sequences_uni.joblib")
print(f"\n✓ sequences_uni  : {len(seq_uni_check)} seq_daily")
for k, v in seq_uni_check.items():
    print(f"   seq={k}: X_train={v['X_train'].shape}  y_train={v['y_train'].shape}"
          f"  | y_test range=[{v['y_test'].min():.4f},{v['y_test'].max():.4f}]")

with open(DATA_PROCESSED / "preprocessing_meta.json") as f:
    meta_check = json.load(f)
print(f"\n✓ meta version   : {meta_check['version']}")
print(f"  cutoff_date    : {meta_check['cutoff_date']}")
print(f"  note           : {meta_check['note']}")
print()
print("🎉 Semua artefak v2 verified — siap lanjut ke modeling_multivariate_new.ipynb")


=== SANITY CHECK v2 ===
✓ scaler_daily   : n_features=2, range=[[ 3937.63 12472.5 ], [ 7318.02 16575.  ]]
✓ scaler_target  : IHSG train range=[3938, 7318]

✓ sequences_dual : 9 kombinasi
  Key            X_daily_te   X_monthly_te     y_test              y_range
  ----------------------------------------------------------------------
  (22, 3)      (485, 22, 2)    (485, 3, 2)     (485,) [0.7775, 1.1738]
  (22, 6)      (485, 22, 2)    (485, 6, 2)     (485,) [0.7775, 1.1738]
  (22, 12)     (485, 22, 2)   (485, 12, 2)     (485,) [0.7775, 1.1738]
  (60, 3)      (485, 60, 2)    (485, 3, 2)     (485,) [0.7775, 1.1738]
  (60, 6)      (485, 60, 2)    (485, 6, 2)     (485,) [0.7775, 1.1738]
  (60, 12)     (485, 60, 2)   (485, 12, 2)     (485,) [0.7775, 1.1738]
  (90, 3)      (485, 90, 2)    (485, 3, 2)     (485,) [0.7775, 1.1738]
  (90, 6)      (485, 90, 2)    (485, 6, 2)     (485,) [0.7775, 1.1738]
  (90, 12)     (485, 90, 2)   (485, 12, 2)     (485,) [0.7775, 1.1738]

── Konsistensi y_test per